![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Day 31 -- Lab 1: Embeddings and Similarity Search

A first look at embeddings, the way today's lecture built them: 480 Kaggle photos of eight everyday kinds (airplane, car, cat, dog, flower, fruit, motorbike, person) go through the pretrained ResNet-18 encoder and through CLIP's image encoder, each photo becoming 512 numbers. You normalise the vectors, build the cosine-similarity table with one matrix product, search for a photo's nearest neighbours, classify every photo by its nearest other photo (against a pixel baseline), draw the space with PCA, then let CLIP classify from sentences (zero-shot) and search from a text query. Nothing is trained.

**Your role:** Fill in the `# Your code here` cells in order. Run every cell, including the ones already written for you.

**Dataset:** Kaggle `prasunroy/natural-images` (about 340 MB), downloaded by `kagglehub` in the setup cell: `natural_images/<class>/*.jpg`, 6,899 photos in eight folders; the setup picks 60 per class spread through each folder (neighbouring files are often near-duplicates). The models download about 650 MB of weights on first use. A CPU runtime is enough (about 3 minutes of embedding); a GPU makes it seconds.

---
# Setup

In [ ]:
# kagglehub fetches the Kaggle dataset; Colab has torch, torchvision, transformers and scikit-learn.
!pip install -q kagglehub

In [ ]:
import os, random
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
from torch import nn
import torch.nn.functional as F
from torchvision.models import resnet18, ResNet18_Weights
from sklearn.decomposition import PCA
from transformers import CLIPModel, CLIPProcessor
random.seed(0)
torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)

In [ ]:
path = kagglehub.dataset_download("prasunroy/natural-images")
ROOT = os.path.join(path, "natural_images")
CLASSES = sorted(os.listdir(ROOT))
paths, labels = [], []
for i, c in enumerate(CLASSES):
    files = sorted(os.listdir(os.path.join(ROOT, c)))
    for f in files[::len(files) // 60][:60]:                  # 60 per class, spread through the folder
        paths.append(os.path.join(ROOT, c, f)); labels.append(i)
labels = torch.tensor(labels)
images = [Image.open(p).convert("RGB") for p in paths]
print(CLASSES)
print(len(images), "photos;", {c: len(os.listdir(os.path.join(ROOT, c))) for c in CLASSES})

def show(indices, titles, ncols=6, size=2.2):
    """Draw photos side by side with titles (used throughout the lab)."""
    fig, axes = plt.subplots(1, len(indices), figsize=(size * len(indices), size))
    for ax, i, t in zip(np.atleast_1d(axes), indices, titles):
        ax.imshow(images[i]); ax.axis("off"); ax.set_title(t, fontsize=9)
    plt.show()

show([int((labels == i).nonzero()[0]) for i in range(8)], CLASSES)

---
# Part 1 -- The encoder's 512 numbers as an embedding

The *Definition*, *In Code: Embed Once, Normalise* and *Similarity Table* slides. Day 29 cut the head off ResNet-18 to get the 512-number vector; today that vector is a point in a space, and cosine similarity says how close two points are.

## Task 1: Embed every photo with ResNet-18

Create `weights = ResNet18_Weights.DEFAULT`, `preprocess = weights.transforms()`, and `encoder = resnet18(weights=weights)` with `encoder.fc = nn.Identity()`, in `eval()` mode on `device`. Write `embed_resnet(images)` that runs the photos in batches of 32 under `torch.no_grad()` (`torch.stack([preprocess(img) for img in batch])`), moves each batch to `device`, and returns one tensor `(N, 512)` on the CPU (`torch.cat` of the batch outputs, `.cpu()`). Run it on `images` as `E_res` and print its shape.

Syntax hint (the shape of the call, not the answer):

```python
def embed_resnet(images, batch_size=32):
    out = []
    with torch.no_grad():
        for i in range(0, len(images), batch_size):
            xb = ...                      # a (B, 3, 224, 224) batch
            out.append(encoder(xb).cpu())
    return torch.cat(out)
```

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert tuple(E_res.shape) == (480, 512), 'embed_resnet: one 512-number row per photo, (480, 512)'
print('Task 1 passed')

## Task 2: Normalise and build the similarity table

Make `E_res = F.normalize(E_res, dim=1)` (every row divided by its length) and `S_res = E_res @ E_res.T`. Print `S_res.shape`, `S_res[0, 0]` (a photo against itself), the similarity between the first two photos of the `dog` class, and between the first `dog` and the first `airplane`. The slide's numbers: 1.0 on the diagonal, same class well above different class, nothing negative.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert tuple(S_res.shape) == (480, 480) and abs(S_res.diagonal() - 1).max() < 1e-4 and S_res[dog[0], dog[1]] > S_res[dog[0], plane[0]], 'S = E @ E.T after F.normalize: a 480 x 480 table with 1.0 on the diagonal; two dogs closer than a dog and an airplane'
print('Task 2 passed')

## Task 3: Image search: the five nearest photos

Write `nearest(S, i, k=5)` that copies row `i` of a similarity table, sets the photo's own entry to `-1`, and returns the indices of the `k` largest values (`topk(k).indices.tolist()`). Run it for the first `dog` photo on `S_res` and draw the query with its five neighbours using `show`, titling each neighbour with its class name and similarity (`f"{CLASSES[labels[j]]} {S_res[q, j]:.2f}"`). Are all five dogs?

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert len(nb) == 5 and q not in nb and all(S_res[q, j] >= S_res[q, nb[-1]] for j in nb), 'nearest: five indices, not the query itself, in order of similarity'
print('Task 3 passed')

## Task 4: A classifier with no training: 1-NN on embeddings

Write `loo_1nn(S)` (leave-one-out): copy `S`, set its diagonal to `-float("inf")` with `fill_diagonal_`, take `argmax(dim=1)` (the most similar *other* photo of every photo), and return the fraction of photos whose neighbour has the same label as they do (`(labels[idx] == labels).float().mean().item()`). Run it on `S_res` and print the accuracy as `acc_res`.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert acc_res > 0.8, '1-NN on ResNet-18 embeddings should pass 0.8 on these eight classes; check the normalisation and the diagonal'
print('Task 4 passed')

## Task 5: The pixel baseline

Build `pix`, a `(480, 1024)` tensor of the photos as 32 x 32 grey images (`np.asarray(img.convert("L").resize((32, 32)), dtype=np.float32).ravel() / 255` for each, stacked with `torch.tensor(np.stack(...))`). Compute the Euclidean distances `D = torch.cdist(pix, pix)` and the 1-NN accuracy of `-D` with `loo_1nn` (a smaller distance is a larger similarity) as `acc_pix`. Compare with Task 4 and with the *Where It Breaks* slide.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert tuple(pix.shape) == (480, 1024) and acc_pix < acc_res, 'pixels: a (480, 1024) tensor whose 1-NN accuracy is below the embedding one'
print('Task 5 passed')

## Task 6: Draw the space

Project `E_res` to two dimensions with `PCA(n_components=2).fit_transform(E_res.numpy())` and draw a scatter plot with one colour per class (loop over the classes, `plt.scatter` of the rows where `labels == i`, `label=c`), a legend, axis labels and a title. Which classes form their own cloud, and which two overlap?

In [ ]:
# Your code here


---
# Part 2 -- CLIP: one space for photos and sentences

The *What CLIP Is*, *In Code: CLIP Zero-Shot* and *Retrieval With Text* slides. CLIP's image encoder gives another 512-number embedding; its text encoder puts sentences in the same space.

## Task 7: Embed every photo with CLIP

Load `clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")` (to `device`, `eval()`) and `processor = CLIPProcessor.from_pretrained(...)`. Write `embed_clip(images)` like Task 1: batches of 32, `inputs = processor(images=batch, return_tensors="pt").to(device)`, `out = clip_model.get_image_features(**inputs)`, then `out = getattr(out, "pooler_output", out)`, collect on the CPU, `torch.cat`, and `F.normalize(..., dim=1)` at the end. Run it as `E_clip`, print the shape and the length of the first row. `getattr(out, "pooler_output", out)` unwraps the tensor: newer versions of `transformers` return an output object, older ones the tensor itself.

Syntax hint (the shape of the call, not the answer):

```python
inputs = processor(images=batch, return_tensors="pt").to(device)
feats = clip_model.get_image_features(**inputs)
feats = getattr(feats, "pooler_output", feats)
```

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert tuple(E_clip.shape) == (480, 512) and abs(E_clip.norm(dim=1) - 1).max() < 1e-4, 'embed_clip: (480, 512) rows of unit length'
print('Task 7 passed')

## Task 8: Search and 1-NN in CLIP's space

Build `S_clip = E_clip @ E_clip.T`, print `acc_clip = loo_1nn(S_clip)` next to `acc_res` and `acc_pix`, and draw the query of Task 3 with its five CLIP neighbours. Also run `nearest` for the fourth `motorbike` photo. Does CLIP's space separate the eight classes better than ResNet-18's?

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert acc_clip > 0.8, '1-NN on CLIP embeddings should pass 0.8'
print('Task 8 passed')

## Task 9: Zero-shot classification from sentences

Make `prompts = [f"a photo of a {c}" for c in CLASSES]`, embed them with `clip_model.get_text_features(**processor(text=prompts, return_tensors="pt", padding=True).to(device))` (unwrap with `getattr(..., "pooler_output", ...)`, `.cpu()`, `F.normalize`) as `T`, then `probs = (100 * E_clip @ T.T).softmax(dim=1)` and `pred = probs.argmax(dim=1)`. Print the zero-shot accuracy `acc_zs = (pred == labels).float().mean().item()` and show four photos of your choice with `f"{CLASSES[pred[i]]} {probs[i].max():.2f}"` as titles. No labelled photo was used.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert tuple(T.shape) == (8, 512) and acc_zs > 0.9, 'eight prompt embeddings; zero-shot accuracy above 0.9 on these classes'
print('Task 9 passed')

## Task 10: Search with a sentence

Embed the queries `["a photo of a red car", "a dog running on grass"]` the same way as the prompts, compute the similarity of every photo to each query (`E_clip @ Tq.T`), and for each query draw the four photos with the highest similarity, titled with the similarity. The gallery has no label for colour or motion; CLIP's captions had.

In [ ]:
# Your code here


---
## Task 11: Reflect

Task 5 against Task 4: name two things pixel distance measures that have nothing to do with what a photo shows, and explain with the *Math* slide why embeddings are compared by cosine rather than by Euclidean distance. In Task 2 no similarity was negative: where does that come from, and why does it not happen with CLIP's embeddings? Task 9 classified 480 photos without a single label: what did the prompts replace, in the words of the *Zero-Shot* slide, and what would you write as the prompt for a dataset of satellite tiles? Task 8 classified by one neighbour; what changes with `k = 5`, and when does kNN lose to the linear probe of Day 30? Finally, which of today's tools would you reach for first in a contest that gives you 2,000 unlabelled photos and asks for duplicates?

*Your answers here*

---
## Conclusion

- **The embedding**: 480 Kaggle photos through the published ResNet-18 encoder, one 512-number row each, normalised to unit length so that one matrix product gives the whole cosine-similarity table.
- **Search and 1-NN without training**: a query's nearest neighbours drawn, every photo classified by its nearest other photo, far above the pixel baseline; the space drawn with PCA.
- **CLIP's shared space**: the same photos embedded by CLIP's image encoder, eight sentences by its text encoder, zero-shot classification as softmax of 100 x cosine, and a sentence used as a search query.

---

Made By **Sattam Altwaim**